# 地域メッシュとは？ Pythonで500mメッシュに人口とPOIを重ねてみた

地域メッシュは、異なる統計を同じ場所のコードで結び付ける共通の区画です。今回は吉祥寺駅周辺で、250m人口を500mへ合計し、POI登録数と結合します。

[方法・出典](../docs/regional-mesh.md) / [検証](../docs/regional-mesh-validation.md) / [記事資料](../docs/regional-mesh-article-handoff.md)

## 1. 緯度・経度をコードにする

第3次は8桁、500mは9桁、250mは10桁。500mは厳密な正方形ではなく、おおよその大きさです。JGD2011の緯度経度でコードを計算し、面積はメートル単位のEPSG:6677で求めます。

In [ ]:
from geoai_open_lab.regional_mesh import encode_mesh, mesh_bounds, run, OUT
from geoai_open_lab.regional_mesh_figures import render
from IPython.display import SVG, display

mesh = encode_mesh(35.70314448120142, 139.57980985190306, digits=9)
mesh, mesh_bounds(mesh)

## 2. 保存済みデータを共通コードで結合

既存の駅中心1km四方に重なる500mメッシュを、切り取らず区画全体で採用します。対象範囲は9区画・約2.353km²へ広がります。人口は2020年、POIは2026年の登録で、同時点の人や店舗ではありません。

固定入力がなければ停止します。ダウンロードは行いません。

In [ ]:
meshes, manifest = run()
meshes[["mesh_code", "population_2020", "poi_count", "area_km2"]]

## 3. 親子関係と結合結果を見る

人口総数は秘匿しない項目です。各500m区画の4個の子メッシュを合計し、POIは緯度経度から同じ9桁コードへ割り当てます。欠けた子メッシュは人口0としません。

In [ ]:
figures = render()
display(SVG(filename=str(OUT / "hierarchy.svg")))
display(SVG(filename=str(OUT / "population_poi.svg")))

## 4. 保存則を確かめる

In [ ]:
assert meshes.population_complete.all()
assert len(meshes) == 9
assert meshes.population_2020.sum() == manifest["population_2020"]
assert meshes.poi_count.sum() == manifest["poi_count"]
{"人口（2020年）": manifest["population_2020"], "POI登録数": manifest["poi_count"]}

## 5. 向く用途と限界

共通コードによる統計の結合、同じ測地系・定義での地域間・時点間の比較に向きます。駅勢圏や商圏を決める境界ではなく、境界をまたぐ施設や集計単位の影響は残ります。POIは実店舗数ではなく、人口も最新人口ではありません。相関や需要は結論にしません。

この実験は250m公表値の合計であり、公式500m配布表そのものとの照合は未実施です。コードと形状は公式の区分規則と配布コードで照合しましたが、公式境界GISとの頂点単位の比較は行っていません。

前記事の範囲感度とは役割を分け、次はEPSGと座標参照系を扱います。